# 01b — Classical Features for Dataset 2

Same feature extraction as Dataset 1, applied to Dataset 2: a static synthetic interbank network (1 444 banks).

Features: degree, weighted degree, betweenness, closeness, eigenvector centrality, PageRank, DebtRank.

Output: `src/data/classical_features/dataset_2/classical_features_dataset2.parquet`

In [ ]:
import os
import sys
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from pathlib import Path

from src.models import (
    degree_centrality,
    betweenness_centrality,
    closeness_centrality,
    eigenvector_centrality,
    weighted_degree,
    debtrank,
    pagerank_centrality,
)

PROJECT_ROOT = find_project_root()
DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'
OUT_FEATURES  = PROJECT_ROOT / 'src' / 'data' / 'classical_features' / 'dataset_2'
OUT_FEATURES.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset 2    : {DATASET2_PATH}')
print(f'Output dir   : {OUT_FEATURES}')

## Load Dataset 2

In [ ]:
def load_dataset2_nodes(dataset_path):
    nodes = pd.read_csv(dataset_path / 'nodes.csv')
    nodes = nodes.reset_index(drop=True)
    nodes['index']  = nodes.index
    nodes['Equity'] = nodes['buffer']
    nodes['Assets'] = nodes['assets']
    return nodes


def load_dataset2_edges(dataset_path, bank_to_idx):
    matrix = pd.read_excel(dataset_path / 'network.xlsx', index_col=0)
    edges_long = matrix.stack().reset_index()
    edges_long.columns = ['source_bank', 'target_bank', 'Weights']
    edges_long = edges_long[edges_long['Weights'] != 0].copy()
    edges_long['Sourceid'] = edges_long['source_bank'].map(bank_to_idx)
    edges_long['Targetid'] = edges_long['target_bank'].map(bank_to_idx)
    edges_long = edges_long.dropna(subset=['Sourceid', 'Targetid'])
    edges_long['Sourceid'] = edges_long['Sourceid'].astype(int)
    edges_long['Targetid'] = edges_long['Targetid'].astype(int)
    return edges_long[['Sourceid', 'Targetid', 'Weights']].reset_index(drop=True)


nodes = load_dataset2_nodes(DATASET2_PATH)
bank_to_idx = dict(zip(nodes['bank'], nodes['index']))
edges = load_dataset2_edges(DATASET2_PATH, bank_to_idx)

print(f'Nodes : {nodes.shape}')
print(f'Edges : {edges.shape}')
display(nodes.head(3))
display(edges.head(3))

## Compute Classical Features

In [ ]:
deg_df  = degree_centrality(edges, nodes)
bet_df  = betweenness_centrality(edges, nodes)
clo_df  = closeness_centrality(edges, nodes)
wdeg_df = weighted_degree(edges, nodes)
pr_df   = pagerank_centrality(edges, nodes, reverse=True)
dr_df   = debtrank(edges, nodes)

features = nodes.copy().rename(columns={'index': 'bank_id'})
for df in (deg_df, bet_df, clo_df, wdeg_df, pr_df, dr_df):
    features = features.merge(df, on='bank_id', how='left')

NETWORK_METRICS = [
    'degree_centrality_total',
    'weighted_degree_in', 'weighted_degree_out', 'weighted_degree_total',
    'betweenness_centrality', 'closeness_centrality',
    'pagerank', 'debtrank',
]
features = features[['bank_id'] + [c for c in NETWORK_METRICS if c in features.columns]]

out_path = OUT_FEATURES / 'classical_features_dataset2.parquet'
features.to_parquet(out_path, index=False)
print(f'[OK] saved {out_path.name}  shape={features.shape}')
print(f'Columns: {features.columns.tolist()}')

## Add Eigenvector Centrality

Computed separately — may fail to converge and is patched into the saved parquet.

In [ ]:
eig_df = eigenvector_centrality(edges, nodes)

out_path = OUT_FEATURES / 'classical_features_dataset2.parquet'
features = pd.read_parquet(out_path)
features = features.merge(eig_df, on='bank_id', how='left')
features.to_parquet(out_path, index=False)
print(f'[OK] added eigenvector_centrality  shape={features.shape}')

## Verify Output

In [ ]:
df = pd.read_parquet(OUT_FEATURES / 'classical_features_dataset2.parquet')
print(f'Shape: {df.shape}')
display(df.head())

feature_cols = [
    'degree_centrality_total', 'weighted_degree_total',
    'betweenness_centrality', 'closeness_centrality',
    'eigenvector_centrality', 'pagerank', 'debtrank',
]
print('\nFeature summary:')
display(df[feature_cols].describe())